# 03. Main Model — Calibrated XGBoost (`xgb-v0.1`)

In this notebook, we evaluate the production model: calibration curve, PR-curve threshold tuning, feature importances, and SHAP explainability.

In [ ]:
import json
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import sys

sys.path.append('..')
from training.features import build_features

# Load production model bundle
bundle_path = Path('../models/xgb-v0.1.pkl')
bundle = joblib.load(bundle_path)
pipeline = bundle['model']
print(f"Loaded production model version: {bundle['version']}")
print("Tuned decision threshold:", bundle['threshold'])
print("Metrics:", json.dumps(bundle['metrics'], indent=2))

### 1. Model Evaluation Artifacts

Inspection of generated reliability, PR, feature importance, and SHAP attribution plots:

In [ ]:
from IPython.display import Image, display

plots_dir = Path('../models/plots')
for plot_name in ['calibration.png', 'pr_curve.png', 'feature_importance.png', 'shap_summary.png', 'shap_waterfall.png']:
    plot_file = plots_dir / plot_name
    if plot_file.exists():
        print(f"=== {plot_name} ===")
        display(Image(filename=str(plot_file)))

### 2. Live Inference Test with Deal Attribution

In [ ]:
# Test payload matching backend deal_features contract §2.1
test_deal = pd.DataFrame([{
    'deal_value': 50000.0,
    'stage': 'proposal',
    'days_in_stage': 12,
    'num_interactions': 8,
    'avg_response_min': 320.0,
    'days_since_last': 3,
    'sentiment_trend': -0.2
}])

win_prob = pipeline.predict_proba(test_deal)[0, 1]
churn_risk = 1.0 - win_prob

print(f"Win Probability: {win_prob:.2%}")
print(f"Churn Risk:      {churn_risk:.2%}")
print(f"Model Version:   {bundle['version']}")